In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
import seaborn as sns
from sklearn.ensemble import IsolationForest

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data=pd.read_csv("/content/drive/MyDrive/spoilage_alert_dataset.csv")
data

,batch_id,date_time,temperature_C,humidity_%,transport_delay_hr,spoilage_probability,alert_flag
0,BATCH_0,2025-01-01 00:00:00,2.351628,66.090376,12,0.007936,0
1,BATCH_1,2025-01-01 06:00:00,2.678474,57.568970,2,0.000000,0
2,BATCH_2,2025-01-01 12:00:00,3.258147,71.118962,13,0.078853,0
3,BATCH_3,2025-01-01 18:00:00,3.017718,51.436514,7,0.000000,0
4,BATCH_4,2025-01-02 00:00:00,4.069736,67.382767,11,0.052722,0
...,...,...,...,...,...,...,...
495,BATCH_495,2025-05-04 18:00:00,6.634299,79.753073,0,0.200217,0
496,BATCH_496,2025-05-05 00:00:00,8.170895,70.873485,6,0.157153,0
497,BATCH_497,2025-05-05 06:00:00,3.131842,68.540920,6,0.033046,0
498,BATCH_498,2025-05-05 12:00:00,3.450512,74.981914,9,0.111329,0


In [ ]:
data.info

<bound method DataFrame.info of       batch_id            date_time  temperature_C  humidity_%  \
0      BATCH_0  2025-01-01 00:00:00       2.351628   66.090376   
1      BATCH_1  2025-01-01 06:00:00       2.678474   57.568970   
2      BATCH_2  2025-01-01 12:00:00       3.258147   71.118962   
3      BATCH_3  2025-01-01 18:00:00       3.017718   51.436514   
4      BATCH_4  2025-01-02 00:00:00       4.069736   67.382767   
..         ...                  ...            ...         ...   
495  BATCH_495  2025-05-04 18:00:00       6.634299   79.753073   
496  BATCH_496  2025-05-05 00:00:00       8.170895   70.873485   
497  BATCH_497  2025-05-05 06:00:00       3.131842   68.540920   
498  BATCH_498  2025-05-05 12:00:00       3.450512   74.981914   
499  BATCH_499  2025-05-05 18:00:00       4.946533   80.010934   

     transport_delay_hr  spoilage_probability  alert_flag  
0                    12              0.007936           0  
1                     2              0.000000           0  
2                    13              0.078853           0  
3                     7              0.000000           0  
4                    11              0.052722           0  
..                  ...                   ...         ...  
495                   0              0.200217           0  
496                   6              0.157153           0  
497                   6              0.033046           0  
498                   9              0.111329           0  
499                  21              0.221540           0  

[500 rows x 7 columns]>

In [ ]:
data.describe()

,temperature_C,humidity_%,transport_delay_hr,spoilage_probability,alert_flag
count,500.000000,500.000000,500.00000,500.000000,500.0
mean,4.015783,70.442409,11.29400,0.096761,0.0
std,1.890867,9.968671,6.77609,0.087634,0.0
min,-3.179601,30.194042,0.00000,0.000000,0.0
25%,2.730550,63.160978,6.00000,0.007752,0.0
50%,3.919174,70.928280,11.00000,0.082628,0.0
75%,5.260819,77.948000,17.00000,0.158526,0.0
max,10.257519,96.695220,23.00000,0.408110,0.0


In [ ]:
data.isna().sum()

,0
batch_id,0
date_time,0
temperature_C,0
humidity_%,0
transport_delay_hr,0
spoilage_probability,0
alert_flag,0


In [ ]:
# 1. Load the Dataset
def load_and_preprocess(file_path):
    df = pd.read_csv(file_path)
    df['date_time'] = pd.to_datetime(df['date_time'])
    return df

In [ ]:
# 2. Threshold-Based Alert System
# This triggers an alert if the spoilage probability exceeds a safe limit
def apply_spoilage_alerts(df, threshold=0.20):
    """
    Sets a flag if spoilage probability is above the defined threshold.
    """
    df['alert_triggered'] = (df['spoilage_probability'] > threshold).astype(int)

    # Define Risk Levels (as previously incorporated logic)
    conditions = [
        (df['spoilage_probability'] < 0.10),
        (df['spoilage_probability'] >= 0.10) & (df['spoilage_probability'] < 0.20),
        (df['spoilage_probability'] >= 0.20)
    ]
    choices = ['SAFE', 'WARNING', 'CRITICAL']
    df['risk_status'] = np.select(conditions, choices, default='UNKNOWN')
    return df

In [ ]:
# Define Risk Levels
conditions = [
    (data['spoilage_probability'] < 0.10),
    (data['spoilage_probability'] >= 0.10) & (data['spoilage_probability'] < 0.20),
    (data['spoilage_probability'] >= 0.20)
]
choices = ['SAFE', 'WARNING', 'CRITICAL']
data['risk_status'] = np.select(conditions, choices, default='UNKNOWN')


In [ ]:
# 3. Machine Learning Anomaly Detection
# Detects batches that have unusual combinations of temperature, humidity, and delay
def detect_anomalies(df):
    features = ['temperature_C', 'humidity_%', 'transport_delay_hr', 'spoilage_probability']

    # Initialize Isolation Forest
    iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

    # Predict (-1 for anomaly, 1 for normal)
    df['anomaly_score'] = iso_forest.fit_predict(df[features])
    df['is_anomaly'] = df['anomaly_score'].apply(lambda x: 1 if x == -1 else 0)

    return df

In [ ]:
# 4. Visualization
def plot_results(df):
    plt.figure(figsize=(14, 6))

    # Subplot 1: Spoilage Probability Over Time with Alerts
    plt.subplot(1, 2, 1)
    sns.lineplot(data=df, x='date_time', y='spoilage_probability', color='gray', alpha=0.5)
    sns.scatterplot(data=df[df['alert_triggered'] == 1], x='date_time', y='spoilage_probability', color='red', label='Alert Triggered')
    plt.title('Spoilage Alerts Over Time')
    plt.xticks(rotation=45)

    # Subplot 2: Anomalies in Temperature vs Humidity
    plt.subplot(1, 2, 2)
    sns.scatterplot(data=df, x='temperature_C', y='humidity_%', hue='is_anomaly', palette={0: 'blue', 1: 'orange'})
    plt.title('Anomaly Detection (Temp vs Humidity)')

    plt.tight_layout()
    plt.show()